In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
# 是否有可用 GPU
print("CUDA available:", torch.cuda.is_available())

# 当前 GPU 名称
if torch.cuda.is_available():
    print("Current GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CUDA available: True
Current GPU: NVIDIA GeForce RTX 3070 Laptop GPU


In [ ]:
# Train basic gameplay model
# Stage 1 - SelfplayEnv with Random opponent

from Client.DeepLearning.Environments.Thesis.SelfPlay import SelfPlayBase
from Client.DeepLearning.Encoders.MainGame.ActionMask.GetActionMask import getActionMask
import os
from DeepLearning.PPO import MaskablePPO
from Client.DeepLearning.Encoders.MainGame.Observation.get_observation_full import getObservationFull

#setupModel = MaskablePPO.load("DeepLearning/Models/ZKA_model/ZKA_SetupStage_1M.zip")
env = SelfPlayBase()
env.selfPlay = False # for random opponents
actionMask = getActionMask
observation = getObservationFull


os.environ["UPDATE_MODELS_DIST"] = "False"
netArchDict = dict(pi=[128, 128, 128], vf=[128, 128, 128])
gamma = 0.99
n_steps = 2048

saveName = "ZKA_FullObservation_GameplayStage_5M_SelfPlay"
savePath = f"DeepLearning/Models/ZKA_model/{saveName}"

model = MaskablePPO("MlpPolicy", env, verbose=1, device=device, policy_kwargs=dict(net_arch=netArchDict), gamma=gamma, n_steps=n_steps, getActionMask=actionMask, getObservation=observation, savePath=savePath, tensorboard_log="./tensorboard_logs_thesis/")

# model = MaskablePPO.load("DeepLearning/Thesis/Setup/Models/SetupRandom/model_332400_5.zip", env=env)
model.savePath = savePath
print("Policy device:", next(model.policy.parameters()).device)
model.learn(total_timesteps=5_000_000, tb_log_name=saveName, reset_num_timesteps=False)

In [ ]:
print(model.observation_space.shape)
print(env.observation_space.shape)

In [ ]:
model.save(savePath)
print(savePath)

In [ ]:
 # Stage 2 - TurnLimitDense + AgainstRandom

from Client.DeepLearning.Environments.Thesis.TurnLimitDense import TurnLimitDense
from Agents.AgentRandom2 import AgentRandom2
from Client.DeepLearning.Encoders.MainGame.ActionMask.GetActionMask import getActionMask
from Client.DeepLearning.Encoders.MainGame.Observation.get_observation_full import getObservationFull
from DeepLearning.PPO import MaskablePPO
import os
os.environ["TURN_LIMIT"] = "30"
os.environ["UPDATE_MODELS_UNIFORM"] = "False"
os.environ["UPDATE_MODELS_DIST"] = "False"

#setupModel = MaskablePPO.load("DeepLearning/Models/ZKA_model/ZKA_SetupStage_1M.zip")
env = TurnLimitDense(players=[
    AgentRandom2("P0", 0),
    AgentRandom2("P1", 1),
    AgentRandom2("P2", 2),
    AgentRandom2("P3", 3)
])
actionMask = getActionMask
observation = getObservationFull

netArchDict = dict(pi=[128, 128, 128], vf=[128, 128, 128])
gamma = 0.99
n_steps = 2048

saveName = "ZKA_Try2_GameplayStage_+3M_Turnlimit"
savePath = f"DeepLearning/Models/ZKA_model/{saveName}"

#model = MaskablePPO("MlpPolicy", env, verbose=1, device=device, policy_kwargs=dict(net_arch=netArchDict), gamma=gamma, n_steps=n_steps, getActionMask=actionMask, getObservation=observation, savePath=savePath, tensorboard_log="./tensorboard_logs_thesis/")

model = MaskablePPO.load("DeepLearning/Models/ZKA_model/model_2330624_177.zip", env=env)
model.savePath = savePath
print("Policy device:", next(model.policy.parameters()).device)
model.learn(total_timesteps=3_000_000, tb_log_name=saveName, reset_num_timesteps=False)

In [ ]:
model.save(savePath)
print(savePath)

In [ ]:
 # Stage 3 - Selfplay/Random start
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
# 是否有可用 GPU
print("CUDA available:", torch.cuda.is_available())

# 当前 GPU 名称
if torch.cuda.is_available():
    print("Current GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


from Client.DeepLearning.Environments.Thesis.SelfPlay import SelfPlayZKA
from Client.DeepLearning.Encoders.MainGame.ActionMask.GetActionMask import getActionMask
from Client.DeepLearning.Encoders.MainGame.Observation.get_observation_full import getObservationFull
from DeepLearning.PPO import MaskablePPO
import os



#setupModel = MaskablePPO.load("DeepLearning/Models/ZKA_model/ZKA_SetupStage_1M.zip")
env = SelfPlayZKA(selfPlay = True, debug = False)
env.denseRewards = True
env.bankTradeRewards = False
actionMask = getActionMask
observation = getObservationFull
os.environ["UPDATE_MODELS_DIST"] = "False"

netArchDict = dict(pi=[128, 128, 128], vf=[128, 128, 128])
gamma = 0.99
n_steps = 2048

saveName = "ZKA_FullObservation_GameplayStage_Selfplay_LinearReward"
savePath = f"DeepLearning/Models/ZKA_model/{saveName}"

#model = MaskablePPO("MlpPolicy", env, verbose=1, device=device, policy_kwargs=dict(net_arch=netArchDict), gamma=gamma, n_steps=n_steps, getActionMask=actionMask, getObservation=observation, savePath=savePath, tensorboard_log="./tensorboard_logs_thesis/")f

model = MaskablePPO.load("DeepLearning/Models/ZKA_model/model_6223354_elo332.zip", env=env)

model.savePath = savePath
print("Policy device:", next(model.policy.parameters()).device)
model.learn(total_timesteps=6_000_000, tb_log_name=saveName, reset_num_timesteps=False)

In [ ]:
# Stage 4 - Selfplay/Random start with MultiModel
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["UPDATE_MODELS_DIST"] = "False"

import torch
# 检查 GPU
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Current GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 1. 导入必要的依赖
# ==========================================
# 确保导入你上一轮创建的那个支持 MultiModel 的环境类
from Client.DeepLearning.Environments.Thesis.SelfPlay import SelfPlayMultiModelZKA

# 导入 Setup 和 Gameplay 的预处理函数（注意不要重复导入 bounds）
from Client.DeepLearning.Encoders.Setup.ActionMask.getSetupActionMask import getSetupActionMask
from Client.DeepLearning.Encoders.Setup.Observation.getObservationSetup import getObservationSetup

from DeepLearning.PPO import MaskablePPO

# ==========================================
# 2. 加载开局阶段的静态模型 (Setup Model)
# ==========================================
print("Loading Setup Model...")
setupModel = MaskablePPO.load("DeepLearning/Models/ZKA_model/ZKA_SetupStage_1M.zip", device=device)

# ==========================================
# 3. 初始化包含 MultiModel 逻辑的环境
# ==========================================
print("Initializing SelfPlayMultiModelZKA Environment...")
# 我们将 setup 的函数和模型传给环境，环境内部会自动生成 4 个 AgentMultiModel
env = SelfPlayMultiModelZKA(
    setupModel=setupModel,
    setup_obs_func=getObservationSetup,
    setup_mask_func=getSetupActionMask,
    selfPlay=True,  # 开启自我博弈
    debug=False     # 建议开启 debug 看看开局是否被正常跳过
)
env.current_step = 0
# ==========================================
# 4. 加载要继续训练的游戏模型 (Gameplay Model)
# ==========================================
print("Loading Gameplay Model for further training...")
# 注意：一定要把 env 传给 load 函数，否则模型不知道在哪里训练！
gameplayModel = MaskablePPO.load("DeepLearning/Models/ZKA_model/ZKA_Multimodel_GameplayStage_Selfplay_LinearReward.zip", env=env, device=device)

# ==========================================
# 5. 配置保存路径并开始训练
# ==========================================
saveName = "ZKA_Multimodel_GameplayStage_Selfplay_LinearReward"
savePath = f"DeepLearning/Models/ZKA_model/{saveName}"

# 确保保存目录存在
os.makedirs(savePath, exist_ok=True)

gameplayModel.savePath = savePath
print("Policy device:", next(gameplayModel.policy.parameters()).device)

print("Starting to learn...")
# 开始训练！reset_num_timesteps=False 保证 Tensorboard 曲线连贯
gameplayModel.learn(total_timesteps=6_000_000, tb_log_name=saveName, reset_num_timesteps=False)

In [ ]:
gameplayModel.save(savePath)
print(savePath)